In [1]:
import moabb
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
from moabb.paradigms import P300
from hoda import HODA, MLSVD
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
import matplotlib.pyplot as plt
import seaborn as sns
from moabb.analysis.plotting import paired_plot, meta_analysis_plot, summary_plot
from sklearn.base import BaseEstimator, ClassifierMixin
from toeplitzlda.classification import ToeplitzLDA
from sklearn.svm import SVC
from moabb.analysis.meta_analysis import (  # noqa: E501
    compute_dataset_statistics,
    find_significant_differences,
)

In [2]:
tmin = 0
tmax=0.7
fmin=0.5
fmax = 16
sfreq = fmax*2

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
    #BNCI2014008(),
    #BNCI2014009()
    EPFLP300()
]
evaluation = WithinSessionEvaluation(
    paradigm=paradigm, datasets=datasets,
    suffix="examples", overwrite=True,
)

In [3]:
import numpy as np

class ToeplitzLDAWrapper(BaseEstimator, ClassifierMixin):

    def fit(self, X, y=None):
        n_epochs, n_channels, n_samples = X.shape
        self.tlda_ = ToeplitzLDA(n_channels=n_channels, data_is_channel_prime=False)
        X = X.reshape(n_epochs, -1)
        return self.tlda_.fit(X, y)

    def decision_function(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.decision_function(X)

    def predict(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.predict(X)

    def predict_proba(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.predict_proba(X)



def lagged_tensor(X, y=None):    
    window = 0.2
    max_lag = 0.5
    
    n_epochs, n_channels, _ = X.shape
    roi0 = int((0 - tmin)*sfreq)
    roi1 = int((window - tmin)*sfreq)
    n_times = roi1-roi0
    n_lags= int(max_lag*sfreq)
    Xt = np.zeros((n_epochs, n_channels, n_times, n_lags))
    for l in range(n_lags):
        Xt[:,:,:,l] = X[:,:,roi0+l:roi1+l]
    return Xt

def reshape(X, y=None):
    return X.reshape((X.shape[0],-1))

In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import SVC

pipelines = dict()

"""
pipelines['tHODA+LDA'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    HODA(max_iter=64, tol=1e-6,
         shrinkage='oas', initialize='identity',
         toeplitz=(1,), rank=5),
    FunctionTransformer(reshape),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
    #SVC(kernel='rbf', C=1)
)
"""

pipelines['lagHODA'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    FunctionTransformer(lagged_tensor),
    MLSVD(modes=(0,),rank=8),
    HODA(max_iter=128, tol=1e-3,
         shrinkage='oas', initialize='identity',
         toeplitz=(1,2), rank=5),
    FunctionTransformer(reshape),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
    #SVC(kernel='rbf', C=1)
)

pipelines['tLDA'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    ToeplitzLDAWrapper()
)


In [5]:
import tensorly as tl
try:
    tl.set_backend('jax')
except AttributeError:
    tl.set_backend('jax')

In [6]:
%prun results = evaluation.process(pipelines )

EPFL P300 dataset-WithinSession:   0%|                                                          | 0/8 [00:00<?, ?it/s]/home/arne/.virtualenvs/hoda-venv/lib/python3.10/site-packages/tensorly/decomposition/_tucker.py:127: Warning: Given only one int for 'rank' instead of a list of 1 modes. Using this rank for all modes.
  warnings.warn(message, Warning)
/home/arne/.virtualenvs/hoda-venv/lib/python3.10/site-packages/tensorly/backend/core.py:1106: UserWarning: In partial_svd: converting to NumPy. Check SVD_FUNS for available alternatives if you want to avoid this.
  warnings.warn('In partial_svd: converting to NumPy.'
EPFL P300 dataset-WithinSession:   0%|                                                          | 0/8 [00:34<?, ?it/s]


TypeError: array() got an unexpected keyword argument 'dtye'

In [ ]:
results

In [ ]:
stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
_=meta_analysis_plot(stats, "lagHODA", "tLDA")

In [ ]:
_=paired_plot(results, "tLDA", "lagHODA")